In [2]:
import carla
import random
import numpy as np
import cv2 as cv
import time

#### Initialisation

In [3]:
client = carla.Client('localhost', 2000)
world = client.get_world()

bpLib = world.get_blueprint_library()
spawnPoints = world.get_map().get_spawn_points()
settings = world.get_settings()
print(settings.synchronous_mode)

False


In [4]:
# tm = client.get_trafficmanager(8000)

#### Generate Barriers

In [5]:
# barrierBP = bpLib.find('static.prop.streetbarrier')
# barrierSpawnPoints = [carla.Transform(carla.Location(x=41.85, y=294.09, z=0), carla.Rotation()),
#              carla.Transform(carla.Location(x=45.85, y=294.09, z=0), carla.Rotation()),
#              carla.Transform(carla.Location(x=190, y=237.01, z=0), carla.Rotation(yaw=90)),
#              carla.Transform(carla.Location(x=190, y=241.01, z=0), carla.Rotation(yaw=90)),
#              carla.Transform(carla.Location(x=190, y=187.61, z=0), carla.Rotation(yaw=90)),
#              carla.Transform(carla.Location(x=190, y=191.61, z=0), carla.Rotation(yaw=90)),
#              carla.Transform(carla.Location(x=0, y=187.68, z=0), carla.Rotation(yaw=-90)),
#              carla.Transform(carla.Location(x=0, y=191.68, z=0), carla.Rotation(yaw=-90))
#              ]
# barriers = []
# for  i in range(len(barrierSpawnPoints)):
#     barrier = world.spawn_actor(barrierBP, barrierSpawnPoints[i])
#     barriers.append(barrier)

#### Generate Vehicles

In [ ]:
vehicleList = []
numVehicles = 5
vehicleSpawnPointList = []
vehicleBP = bpLib.find('vehicle.seat.leon')
for i in range(numVehicles):
    location = random.choice(spawnPoints)
    while location in vehicleSpawnPointList:
        print(location)
        location = random.choice(spawnPoints)
    vehicleSpawnPointList.append(location)
    vehicle = world.spawn_actor(vehicleBP, location)
    vehicleList.append(vehicle)

location = random.choice(spawnPoints)
while location in vehicleSpawnPointList:
    location = random.choice(spawnPoints)
mainVehicle = world.spawn_actor(vehicleBP, location)

spectator = world.get_spectator()
transform = carla.Transform(mainVehicle.get_transform().transform(carla.Location(x=-4, z=2.5)), carla.Rotation())
spectator.set_transform(transform)

cameraBP = bpLib.find('sensor.camera.rgb')
cameraBP.set_attribute('image_size_x', '1920')
cameraBP.set_attribute('image_size_y', '1080')
cameraInitTrans = carla.Transform(carla.Location(x=0.4, z=1.6))
camera = world.spawn_actor(cameraBP, cameraInitTrans, attach_to=mainVehicle)

#### Generate Walkers

In [7]:
walkerList = []
walkerControllerList = []
numWalkers = 5
walkerSpawnPointList = []

walkerBP = bpLib.find('walker.pedestrian.0002')
walkerControllerBP = bpLib.find('controller.ai.walker')
for i in range(numWalkers):
    spawnPoint = carla.Transform()
    location = world.get_random_location_from_navigation()
    while location in walkerSpawnPointList:
        location = world.get_random_location_from_navigation()
        print(location)
    walkerSpawnPointList.append(location)
    if location != None:
        spawnPoint.location = location
        walker = world.spawn_actor(walkerBP, spawnPoint)
        walkerList.append(walker)

for i in range(numWalkers):
    controller = world.spawn_actor(walkerControllerBP, carla.Transform(), walkerList[i])
    walkerControllerList.append(controller)

world.wait_for_tick()
print(len(walkerList))
print(len(walkerControllerList))
for i in range(len(walkerControllerList)):
    print(i, walkerControllerList[i].is_alive, walkerControllerList[i].parent)

5
5
0 True Actor(id=119, type=walker.pedestrian.0002)
1 True Actor(id=120, type=walker.pedestrian.0002)
2 True Actor(id=121, type=walker.pedestrian.0002)
3 True Actor(id=122, type=walker.pedestrian.0002)
4 True Actor(id=123, type=walker.pedestrian.0002)


#### Start

In [8]:
def cameraCallBack(image, dataDict):
    dataDict['image'] = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))

In [ ]:
imageW = cameraBP.get_attribute('image_size_x').as_int()
imageH = cameraBP.get_attribute('image_size_y').as_int()

cameraData = {'image': np.zeros((imageH, imageW, 4))}

camera.listen(lambda image: cameraCallBack(image, cameraData))

: 

In [ ]:
for i in range(numVehicles):
    vehicleList[i].set_autopilot(True)
mainVehicle.set_autopilot(True)

for i in range(numWalkers):
    walkerControllerList[i].start()
    walkerControllerList[i].go_to_location(world.get_random_location_from_navigation())
    walkerControllerList[i].set_max_speed(float(walkerBP.get_attribute('speed').recommended_values[1]))

cv.namedWindow('RGB Camera', cv.WINDOW_AUTOSIZE)
cv.imshow('RGB Camera', cameraData['image'])
cv.waitKey(1)
t = time.time()
autolist = []
currentLocation = []
for i in range(numVehicles):
    currentLocation.append([round(vehicleList[i].get_location().x, 3), round(vehicleList[i].get_location().y, 3), round(vehicleList[i].get_location().z, 3)])
    autolist.append(True)
mainVehicleLocation = round(mainVehicle.get_location().x, 3), round(mainVehicle.get_location().y, 3), round(mainVehicle.get_location().z, 3)
mainVehicleAuto = True
print(currentLocation)

while True:
    world.wait_for_tick()
    for i in range(numVehicles):
        if not autolist[i]:
            autolist[i] = True
            vehicleList[i].set_autopilot(True)
            print(f'Enable automous driving in vehicle {i}')
    if not mainVehicleAuto:
        mainVehicleAuto == True
        print('Enable automous driving in main vehicle')
    
    if time.time() - t > 5:
        t = time.time()
        for i in range(5):
            loc = [round(vehicleList[i].get_location().x, 3), round(vehicleList[i].get_location().y, 3), round(vehicleList[i].get_location().z, 3)]
            # print(loc)
            if loc == currentLocation[i]:
                vehicleList[i].set_autopilot(False)
                autolist[i] = False
                print(f'Disable automous driving in vehicle {i}')
    
            currentLocation[i] = loc
        loc = [round(vehicleList[i].get_location().x, 3), round(vehicleList[i].get_location().y, 3), round(vehicleList[i].get_location().z, 3)]
        if loc == mainVehicleLocation:
            mainVehicle.set_autopilot(False)
            mainVehicleAuto = False
            mainVehicleLocation = loc


    cv.imshow('RGB Camera', cameraData['image'])

    # print(vehicle.is_alive, vehicle.get_location())

    if cv.waitKey(1) == ord('q'):
        for i in range(numWalkers):
            walkerControllerList[i].stop()
            walkerControllerList[i].destroy()
            walkerList[i].destroy()
        for i in range(numVehicles):
            vehicleList[i].set_autopilot(False)
            vehicleList[i].destroy()
        mainVehicle.destroy()
        break

cv.destroyAllWindows()

[[-3.68, 121.21, 0.215], [41.39, 212.98, 0.215], [55.41, 105.55, 0.215], [173.87, 105.55, 0.215], [25.53, 105.55, 0.215]]
Disable automous driving in vehicle 1
Enable automous driving in vehicle 1


In [ ]:
# for i in range(len(barrierSpawnPoints)):
#     barriers[i].destroy()